<a href="https://colab.research.google.com/github/zakyalkhair/alfagift-sentiment-analysis/blob/main/script-py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google_play_scraper
!pip install textblob
from google_play_scraper import app
import pandas as pd
import numpy as np
import sklearn
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
import textblob
#from wordcloud import WordCloud
from pathlib import Path
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report, accuracy_score

import pickle
import re
import time
import datetime                              # access to %%time, for timing individual notebook cells
import os
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline
%config InlineBackend.figure_format='retina'

sns.set_theme()
plt.rcParams["figure.figsize"] = (15,10)

In [ ]:
#Android App NHS link from Google Play at
#https://play.google.com/store/apps/details?id=com.alfamart.alfagift
#The apps ID found in the link after com.alfamart.alfagift
#The apps name on Google Play titled: Alfagift
#Dated Feb 19, 2026: number of reviews 305K

from google_play_scraper import app, Sort, reviews_all

alfagift_reviews = reviews_all(
    'com.alfamart.alfagift',
    sleep_milliseconds=0, # defaults to 0
    lang='id', # defaults to 'id'
    sort=Sort.NEWEST, # defaults to Sort.MOST_RELEVANT
)

In [ ]:
#Save the Alfagift Apps reviews into dataframe
df_alfagift = pd.DataFrame(np.array(alfagift_reviews),columns=['content'])
df_alfagiftrev = df_alfagift.join(pd.DataFrame(df_alfagift.pop('content').tolist()))
df_alfagiftrev.to_csv(r'df_alfagift.csv', index=False)
df_alfagift.head()

In [ ]:
print(df_alfagiftrev.reviewCreatedVersion.unique())
print(df_alfagiftrev.reviewCreatedVersion.nunique())

In [ ]:
#We do not need data for column reviewId, userName, userImage so
#we will show only these column
df_alfagiftrev.loc[:,["content","score","thumbsUpCount", "reviewCreatedVersion", "at", "replyContent", "repliedAt"]]

In [ ]:
!pip install emoji

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 200)
#from pandas_profiling import ProfileReport
import regex
import nltk
import wordcloud
import textblob

from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from wordcloud import WordCloud, STOPWORDS
from textblob import TextBlob

import string
import re
import emoji

In [ ]:
#Creating polarity on the column: content (review from the apps) using TextBlob
#Read https://textblob.readthedocs.io/en/dev/quickstart.html

from textblob import TextBlob
# Ensure 'content' column has no None values and is of string type
df_alfagiftrev['content'] = df_alfagiftrev['content'].fillna('').astype(str)
df_alfagiftrev['sentiment_polarity'] = df_alfagiftrev['content'].apply(lambda x: TextBlob(x).polarity)
df_alfagiftrev['sentiment_subjective'] = df_alfagiftrev['content'].apply(lambda x: TextBlob(x).subjectivity)

In [ ]:
#df_nhsrev.loc[:,["content","score","sentiment_polarity", "sentiment_subjective","at"]]
df_alfagiftrev.loc[:,["content","score","sentiment_polarity", "sentiment_subjective"]]

In [ ]:
#Check number of reviews scores
df_alfagiftrev['score'].value_counts()

In [ ]:
plt.hist(df_alfagiftrev['score'])
plt.show()

In [ ]:
#DATA PREPROCESSING
# Lower casing

# Change the reviews type to string
df_alfagiftrev['content'] = df_alfagiftrev['content'].astype(str)# Before lowercasing
# Before lowercasing
print(df_alfagiftrev['content'][22442])

In [ ]:
#Lowercase all reviews to see the difference
df_alfagiftrev['content']= df_alfagiftrev['content'].apply(lambda x: x.lower())
print(df_alfagiftrev['content'][22442])

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
#edited from https://www.tensorscience.com/nlp/sentiment-analysis-tutorial-in-python-classifying-reviews-on-movies-and-products
import string

df_alfagiftrev['wordCount'] = [len(review.split()) for review in df_alfagiftrev['content']]

df_alfagiftrev['uppercaseCharCount'] = [sum(char.isupper() for char in review) \
                              for review in df_alfagiftrev['content']]

df_alfagiftrev['specialCharCount'] = [sum(char in string.punctuation for char in review) \
                            for review in df_alfagiftrev['content']]

In [ ]:
df_alfagiftrev.loc[:,["content","score","sentiment_polarity", "sentiment_subjective","wordCount","uppercaseCharCount","specialCharCount"]]

In [ ]:
#Removing stopwords
#Using nltk
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize, sent_tokenize
from nltk import FreqDist
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer


from wordcloud import WordCloud, STOPWORDS
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from string import punctuation

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
# function to plot most frequent terms
def freq_words(x, terms = 30):
  all_words = ' '.join([text for text in x])
  all_words = all_words.split()

  fdist = FreqDist(all_words)
  words_df = pd.DataFrame({'word':list(fdist.keys()), 'count':list(fdist.values())})

  # selecting top 20 most frequent words
  d = words_df.nlargest(columns="count", n = terms)
  plt.figure(figsize=(20,5))
  ax = sns.barplot(data=d, x= "word", y = "count")
  ax.set(ylabel = 'Count')
  plt.show()

In [ ]:
freq_words(df_alfagiftrev['content'])

In [ ]:
#check if there is any special character
alphabet = string.ascii_letters+string.punctuation
print(df_alfagiftrev.content.str.strip(alphabet).astype(bool).any())

extracted_emojis=[]

def extract_emojis(s):
    expe = re.compile('[\U00010000-\U0010ffff]', flags=re.UNICODE)
    return expe.findall(s)
    return expe.sub(r'',s)

for y in df_alfagiftrev['content']:
    #print(str(extract_emojis(y)))
    extracted_emojis.append(str(extract_emojis(y)))

print(extracted_emojis)

In [ ]:
df_alfagiftrev.loc[:,["content", "score","sentiment_polarity", "sentiment_subjective","wordCount","uppercaseCharCount","specialCharCount"]]

In [ ]:
#Add a column name polarity_rating from changing the score of the review into 3 labels: Pos, Negative Neutral
df_alfagiftrev['sentiment_rating'] = df_alfagiftrev['score'].apply(lambda x: 'Positive' if x > 3 else('Neutral' if x == 3  else 'Negative'))

In [ ]:
df_alfagiftrev.loc[:,["content","score","sentiment_polarity", "sentiment_subjective","sentiment_rating"]]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(15, 10))
sns.scatterplot(x=df_alfagiftrev['sentiment_polarity'], y=df_alfagiftrev['sentiment_subjective'],
                hue = df_alfagiftrev['sentiment_rating'], edgecolor='white', palette="pastel")
plt.title("Google Play Store Alfagift Track and Trace Reviews Sentiment Analysis", fontsize=20)
plt.show()

In [ ]:
#Print the extracted emojis review column (content) before stopwords
df_alfagiftrev.to_csv(r'df_alfagiftrev_before_stopwords_sentiment_rating.csv', index = False)

In [ ]:
import nltk
from nltk.corpus import stopwords

# Download Indonesian stopwords if not already downloaded
try:
    stopwords.words('indonesian')
except LookupError:
    nltk.download('stopwords')

stop = stopwords.words('english') + ["someone","still","would","need", "gak", "ga", "yg", "aja", "nya", "gk" ,",", "."]
stop.extend(stopwords.words('indonesian'))
print(stop)

In [ ]:
print(len(stop))

In [ ]:
df_alfagiftrev_stopwords = df_alfagiftrev.loc[:,["content","score","sentiment_polarity", "sentiment_subjective","sentiment_rating"]]

In [ ]:
# Exclude stopwords
df_alfagiftrev_stopwords['tweet_without_stopwords'] = df_alfagiftrev_stopwords['content'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))
df_alfagiftrev_stopwords.loc[:,["content","tweet_without_stopwords"]]

In [ ]:
# pat = r'\b(?:{})\b'.format('|'.join(stop))
# The lines below are removed as the stopword removal is already handled by cell 37f3557d
# df_alfagiftrev_stopwords['tweet_without_stopwords'] = df_alfagiftrev_stopwords['content'].str.replace(pat, '')
# df_alfagiftrev_stopwords['tweet_without_stopwords'] = df_alfagiftrev_stopwords['tweet_without_stopwords'].str.replace(r'\s+', ' ')
print(df_alfagiftrev_stopwords)

In [ ]:
df_alfagiftrev_stopwords.loc[:,["content","tweet_without_stopwords"]]

In [ ]:
#Save into csv after applying stopwords
df_alfagiftrev_stopwords.to_csv(r'df_alfagiftrev_after_stopwords_sentiment_rating2.csv', index = False)

In [ ]:
#Count after stopwords
#edited from https://www.tensorscience.com/nlp/sentiment-analysis-tutorial-in-python-classifying-reviews-on-movies-and-products
import string

df_alfagiftrev_stopwords['wordCount'] = [len(review.split()) for review in df_alfagiftrev_stopwords['content']]
df_alfagiftrev_stopwords['wordCount_after_stopwords'] = [len(review.split()) for review in df_alfagiftrev_stopwords['tweet_without_stopwords']]


In [ ]:
df_alfagiftrev_stopwords.loc[:,["content","score","sentiment_polarity", "sentiment_rating","wordCount","tweet_without_stopwords", "wordCount_after_stopwords"]]

In [ ]:
# function to plot most frequent terms
def freq_words(x, terms = 30):
  all_words = ' '.join([text for text in x])
  all_words = all_words.split()

  fdist = FreqDist(all_words)
  words_df = pd.DataFrame({'word':list(fdist.keys()), 'count':list(fdist.values())})

  # selecting top 20 most frequent words
  d = words_df.nlargest(columns="count", n = terms)
  plt.figure(figsize=(20,5))
  ax = sns.barplot(data=d, x= "word", y = "count")
  ax.set(ylabel = 'Count')
  plt.show()

In [ ]:
freq_words(df_alfagiftrev_stopwords['tweet_without_stopwords'])

In [ ]:
newStopWords = ["'",".",",", "someone","still","would","need",""]

#from nltk.corpus import stopwords
#stoplist = stopwords.words('english') + ['though']

#NOT YET DONE ==== 24 May 2022

In [ ]:
#Stemming

#Stemming function chops off the end of the word
#and transform the word into its root form.
#All suffixes like -s, -es, -ed, -ing are removed.

def stemming(x):
    st = PorterStemmer()
    if x is not None:
       for word in x.split():
           st.stem(word)

df_alfagiftrev_stopwords['tweet_without_stopwords'].apply(lambda x:stemming(x))
print(df_alfagiftrev_stopwords['tweet_without_stopwords'][22439])

In [ ]:
#DROP NA
df_alfagiftrev_stopwords.dropna(inplace=True)
df_alfagiftrev_stopwords.info()

In [ ]:
df_alfagiftrev_stopwords.loc[:,["content","score","sentiment_rating","wordCount","tweet_without_stopwords", "wordCount_after_stopwords"]]